# Getting started

**APSG** defines several new python classes to easily manage, analyze and visualize orientation structural geology data. There are several classes to work with orientation data, namely `Vector3` for vectorial data and `Lineation`, `Foliation` for axial data

## Basic usage

**APSG** module could be imported either into its own namespace or into the active one for easier interactive work.

In [ ]:
from apsg import *

### Basic operations with vectors in 2D and 3D
Instance of vector object `Vector3` could be created by passing three arguments corresponding to 3 components to function `vec`:

In [ ]:
u = vec(1, -2, 3)
v = vec(-2, 1, 1)

Alternative ways to create a vector are to pass a single iterable object as list, tuple or array, or to provide geological orientation according to **APSG** notation (default is *direction of dip* and *dip angle* for planar and *trend* and *plunge* for linear features)

In [ ]:
coords = (-2, 2, 3)
a = vec(coords)
b = vec(120, 60)
print(a, b)

The instance of a 2D vector `Vector2` could be created passing 2 arguments corresponding to 2 components or a single iterable of components to the function `vec2`:

In [ ]:
k = vec2(1, 1)
coords = (-1, 2)
l = vec2(coords)
print(k, l)

Alternatively, single argument is interpreted as direction in degrees (0-top, 90-right, 180-down, 270-left)

In [ ]:
m = 5 * vec2(120)
m

For common vector operations we can use standard mathematical operators or special methods using dot notation

In [ ]:
u + v

In [ ]:
u - v

In [ ]:
3*u - 2*v

Its magnitude or length is most commonly defined as its Euclidean norm and could be calculated using `abs`

In [ ]:
abs(v)

In [ ]:
abs(u + v)

For *dot product* we can use the `dot` method or the `@` operator 

In [ ]:
u.dot(v)

In [ ]:
u @ v

For *cross product* we can use the `cross` method

In [ ]:
u.cross(v)

To project vector ``u`` onto vector ``v`` we can use method ``proj``

In [ ]:
u.proj(v)

To find angle (in degrees) between to vectors we use method ``angle``

In [ ]:
u.angle(v)

The ``rotate`` method rotates a vector around another vector. For example, to rotate vector ``u`` around vector ``v`` for 45°

In [ ]:
u.rotate(v, 45)

## Classes Lineation and Foliation
To work with orientation data in structural geology, **APSG** provides two classes: ``Foliation`` to represent planar features and ``Lineation`` to represent linear features. Both classes support all `Vector3` methods and operators, but it should be noted, that `dot` and `angle` respect their axial nature, i.e. angle between two lineations can't be bigger than 90 degrees.

To create instance of `Lineation` or `Foliation`, we can use functions `lin` and `fol`. Arguments have similar syntax to `vec`.

In [ ]:
lin(120, 60), fol(216, 62)

We can also cast `Vector3` instance to `Foliation` or `Lineation`

In [ ]:
lin(u), fol(u)

### Vector methods for Lineation and Foliation


To find angle between two linear or planar features we can use method `angle`

In [ ]:
l1 = lin(110, 40)
l2 = lin(160, 30)
l1.angle(l2)

In [ ]:
p1 = fol(330, 50)
p2 = fol(250, 40)
p1.angle(p2)

We can use *cross product* to construct a planar feature defined by two linear features

In [ ]:
l1.cross(l2)

or to construct linear feature defined by intersection of two planar features

In [ ]:
p1.cross(p2)

*Cross product* of planar and linear features could be used to construct a plane defined by a linear feature and the normal of a planar feature

In [ ]:
l2.cross(p2)

or to find the perpendicular linear feature on a given plane

In [ ]:
p2.cross(l2)

To rotate structural features we can use method ``rotate``

In [ ]:
p2.rotate(l2, 45)

## Notations
By default, **APSG** displays planar and linear features using *dip direction and dip* notation (`dd`). Two more conventions are available: right-hand-rule strike and dip (`rhr`), and a textual quadrant notation (`quadrant`), e.g. `N30E,40NW`. The active notation is controlled by `apsg_conf.notation` and can be changed permanently by assigning to it, or temporarily using the `apsg_conf_context` context manager.

In [ ]:
f = fol(210, 40)
f

Switching notation only changes how a feature is displayed - the underlying orientation is unchanged:

In [ ]:
with apsg_conf_context(notation="rhr"):
    print(f)

with apsg_conf_context(notation="quadrant"):
    print(f)

Quadrant notation is a text format, so `Foliation` and `Lineation` (and anything built from them, like `Pair`, `Fault` and `Cone`) can also be constructed directly from quadrant strings:

In [ ]:
fol("N30E,40NW"), lin("N45E,30")

## Classes Pair and Fault
To work with paired orientation data like foliations and lineations or fault data in structural geology, **APSG** provides the ``Pair`` base class and the derived ``Fault`` class. Both classes are instantiated providing dip direction and dip of planar and linear measurements, which are automatically orthogonalized. If misfit is too high, warning is raised. The `Fault` class expects one more argument providing sense of movement information, either 1 or -1 for normal/reverse movement.

To create instance of `Pair`, we have to pass two arguments for planar and two arguments for linear features following geological notation to function `pair`:

In [ ]:
p = pair(120, 40, 162, 28)
p

In [ ]:
p.misfit

Planar and linear features are accessible using `fol` and `lin` properties

In [ ]:
p.fol, p.lin

To rotate a ``Pair`` instance we can use the ``rotate`` method

In [ ]:
p.rotate(lin(45, 10), 60)

Instantiation of ``Fault`` class is similar, we just have to provide an argument to define the sense of movement

In [ ]:
f = fault(120, 60, 110, 58, 1)  # 1 for normal fault
f

Note the change in sense of movement after ``Fault`` rotation

In [ ]:
f.rotate(lin(45, 10), 60)

For simple fault analyses the ``Fault`` class also provides ``p``, ``t``, ``m``, and ``d`` properties to get PT-axes, kinematic plane and dihedra separation plane

In [ ]:
f.p, f.t, f.m, f.d

## Class Cone
General feature type to store small or great circles as a cone. It could be defined by `axis`, `secant` and `revolving angle` or by ``axis`` and ``apical angle``. The revolving angle is by default 360° defining full cone. For segments of small circles, the `revolving angle` could be different.

In [ ]:
c = cone(lin(140, 50), lin(140, 75))
c

To create small circle segment, you can provide `revolving angle`

In [ ]:
c = cone(lin(90, 70), lin(45,30), 115)

The tips of small circle segments could be obtained from the cone properties ``secant`` and ``rotated_secant``

In [ ]:
lin(c.secant), lin(c.rotated_secant)

To define cone using `apical angle`, use `axis` and number arguments. Note, that `revolving angle` is 360 by default

In [ ]:
c = cone(lin(140, 50), 25)
c

In [ ]:
c.revangle

## Feature sets
*APSG* provides several classes to process, analyze, and visualize sets of data. There are e.g. `vecset`, `linset` and `folset` classes to store groups of ``vec``, ``lin``, and ``fol`` objects. All these feature sets are created from homogeneous list of data with optional `name` attribute.

In [ ]:
v = vecset([vec(120,60), vec(116,50), vec(132,45), vec(90,60), vec(84,52)], name='Vectors')
v

In [ ]:
l = linset([lin(120,60), lin(116,50), lin(132,45), lin(90,60), lin(84,52)], name='Lineations')
l

In [ ]:
f = folset([fol(120,60), fol(116,50), fol(132,45), fol(90,60), fol(84,52)], name='Foliations')
f

To simplify interactive group creation, you can use function ``G``

In [ ]:
g = G([lin(120,60), lin(116,50), lin(132,45), lin(90,60), lin(84,52)], name='L1')
g

Method ``len`` returns number of features in group

In [ ]:
len(v)

Most of the `vec`, `lin` and `fol` methods could be used for feature sets as well. For example, to measure angles between all features in group and another feature, we can use method `angle`:

In [ ]:
l.angle(lin(110,50))

To rotate all features in group around another feature, we can use method ``rotate``

In [ ]:
lr = l.rotate(lin(150, 30), 45)

To show data in list you can use ``data`` property

In [ ]:
l.data

In [ ]:
lr.data

Method ``R`` returns the resultant of all features in set. Note that `Lineation` and `Foliation` are axial in nature, so resultant vector is not reliable. Check the orientation tensor analysis below.

In [ ]:
v.R()

In [ ]:
lin(v.R())

There are several methods to infer spherical statistics as spherical variance, Fisher's statistics, confidence cones on data etc.

In [ ]:
l.var()

In [ ]:
v.fisher_statistics()

In [ ]:
v.fisher_cone()

In [ ]:
v.fisher_cone_csd()

In [ ]:
v.delta()

In [ ]:
v.rdegree()

To calculate orientation tensor of all features in group, we can use `ortensor` method.

In [ ]:
v.ortensor()